# Лекция 13. Реляционные базы данных

Файл хранит байты, а база данных поддерживает общую модель записей, связей и допустимых изменений. На примере клиентов и заказов разберём, как выразить правила в схеме, получить данные одним запросом и не оставить половину операции после ошибки. SQLite позволит выполнить всё локально, а SQLModel в конце покажет, как те же понятия выглядят через Python-объекты.

## Цели

После лекции вы сможете:

- моделировать сущности таблицами и связывать строки ключами;
- задавать ограничения `NOT NULL`, `UNIQUE`, `CHECK` и `FOREIGN KEY`;
- писать параметризованные CRUD-запросы;
- правильно работать с `NULL`;
- применять `INNER JOIN`, `LEFT JOIN`, группировку и агрегаты;
- объединять изменения транзакцией;
- понимать назначение и стоимость индекса;
- работать с SQLite через `sqlite3` Python 3.14;
- объяснять роли engine, session и модели в ORM;
- распознавать запросы, которые выглядят разумно, но меняют смысл данных.

## Перед началом

Нужны контейнеры, классы, исключения и контекстные менеджеры. На модель и ограничения заложено около 20 минут, на базовый SQL и `NULL` — 20 минут, на `JOIN` и агрегаты — 20 минут, на транзакции — 15 минут, на индексы, Python API и ORM — 15 минут. Контринтуитивные примеры разбираются рядом с соответствующим правилом и собираются ещё раз в конце.

Все основные примеры используют базу SQLite в памяти. После закрытия соединения она исчезнет, поэтому ноутбук не создаёт учебные файлы в репозитории.

## Зачем база, если есть CSV и JSON

Файл удобен, когда набор читается или переписывается целиком одним процессом. Приложению с клиентами и заказами нужны другие операции:

- найти одну запись по ключу;
- изменить её, не переписывая весь набор;
- запретить заказ несуществующего клиента;
- согласованно выполнить несколько изменений;
- обслуживать несколько обращений;
- ускорить частые запросы.

СУБД управляет хранением и предоставляет язык запросов, ограничения, транзакции и индексы. Она не угадывает предметную модель: плохая схема останется плохой и под хорошей СУБД.

## Таблицы, строки и столбцы

Таблица описывает множество однотипных фактов. Строка `customers` представляет одного клиента, столбцы — его атрибуты. Строка `orders` представляет отдельный заказ.

Полезный тест именования: одна строка должна отвечать на вопрос «один что?». Если столбец `orders` содержит строку с перечислением всех заказов клиента, мы потеряли отдельные строки и усложнили фильтрацию, ограничения и агрегаты. Если в заказе повторяются email и имя клиента, при переименовании придётся согласованно менять множество копий.

## Отделяем сущности и связи

Клиент существует независимо от конкретного заказа, поэтому ему нужна собственная таблица. В заказе хранится только ссылка `customer_id`. Получается связь «один ко многим»: у одного клиента много заказов, каждый заказ принадлежит одному клиенту.

Нормализация уменьшает противоречивые копии фактов. Но это не соревнование по числу таблиц. Адрес доставки, зафиксированный на момент заказа, иногда правильно хранить именно в заказе: будущая смена адреса клиента не должна переписать историю. Решение следует из смысла факта и его времени жизни.

## Первичный и внешний ключ

**Первичный ключ** однозначно определяет строку. Email выглядит естественным ключом клиента, но может измениться и неудобен для ссылок, поэтому используем технический целочисленный `id`, а уникальность email задаём отдельно.

**Внешний ключ** `orders.customer_id` ссылается на `customers.id`. Он запрещает осиротевшую ссылку. Действие `ON DELETE CASCADE` означает, что удаление клиента удалит его заказы; это сильное бизнес-решение, а не обязательная настройка каждой связи. Иногда правильнее запретить удаление или сохранить запись с архивным статусом.

## Ограничения как исполняемые правила

- `NOT NULL` запрещает отсутствие значения.
- `UNIQUE` запрещает повтор ключа.
- `CHECK` проверяет условие строки.
- `FOREIGN KEY` проверяет существование связанной строки.
- `PRIMARY KEY` задаёт идентичность строки.

Проверка только в форме или Python-функции недостаточна: данные могут прийти из другого скрипта, миграции или административной команды. База должна защищать базовые инварианты независимо от пути записи. Сложные правила, зависящие от внешних сервисов, всё равно остаются в приложении.

## Соединение SQLite в Python 3.14

Внешние ключи SQLite нужно явно включать для каждого соединения и до начала транзакции. Сначала подключаемся в режиме SQLite autocommit, выполняем `PRAGMA`, затем включаем рекомендуемое PEP 249-управление транзакциями. `sqlite3.Row` позволяет обращаться к столбцам по имени.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:", autocommit=True)
connection.execute("PRAGMA foreign_keys = ON")
assert connection.execute("PRAGMA foreign_keys").fetchone()[0] == 1
connection.autocommit = False
connection.row_factory = sqlite3.Row

> **Появилось в Python 3.12.** Атрибут и параметр `Connection.autocommit` стали рекомендуемым способом управлять транзакциями `sqlite3`. В Python 3.14 значение по умолчанию всё ещё `LEGACY_TRANSACTION_CONTROL`, поэтому в учебном коде режим задаётся явно. `autocommit=False` означает PEP 249-поведение с явными `commit()` и `rollback()`; это не то же самое, что низкоуровневый SQLite autocommit.

## DDL: описываем схему

Команды определения данных создают таблицы и ограничения. Денежную сумму храним целым числом копеек: так сравнение и сложение не наследуют погрешности двоичного `float`. Статус ограничен небольшим перечислением. Для большой изменяемой системы статусы иногда выносят в справочник, но здесь `CHECK` проще и честнее.

In [ ]:
schema = """
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    name TEXT NOT NULL CHECK (length(trim(name)) > 0)
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount_cents INTEGER NOT NULL CHECK (amount_cents >= 0),
    status TEXT NOT NULL CHECK (status IN ('new', 'paid', 'cancelled')),
    comment TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(id) ON DELETE CASCADE
);
"""
try:
    connection.executescript(schema)
    connection.commit()
except Exception:
    connection.rollback()
    raise

## DML: изменяем данные

`INSERT`, `UPDATE` и `DELETE` меняют строки, `SELECT` читает результат. Python-значения передаются отдельно через placeholders. Это не просто экранирование: драйвер отличает структуру SQL от значения и кодирует тип по правилам базы.

Placeholder заменяет только значение. Имя таблицы, столбца или направление сортировки через `?` передать нельзя; допустимые элементы структуры выбирают из заранее заданного набора в коде.

In [ ]:
customers = [
    ("anna@example.test", "Анна"),
    ("boris@example.test", "Борис"),
    ("vera@example.test", "Вера"),
]
connection.executemany(
    "INSERT INTO customers (email, name) VALUES (?, ?)",
    customers,
)
connection.executemany(
    "INSERT INTO orders (customer_id, amount_cents, status, comment) VALUES (?, ?, ?, ?)",
    [
        (1, 900, "new", None),
        (1, 2500, "paid", "доставка вечером"),
        (2, 1200, "paid", None),
    ],
)
connection.commit()

## `SELECT`: формулируем результат

Запрос декларативен: мы описываем нужный набор, а план выполнения выбирает СУБД. Основные части читаются почти как конвейер:

- `SELECT` — какие выражения вернуть;
- `FROM` — из каких таблиц;
- `WHERE` — какие исходные строки оставить;
- `ORDER BY` — как упорядочить;
- `LIMIT` — сколько строк вернуть.

Без `ORDER BY` порядок строк не гарантирован, даже если маленькая таблица несколько раз случайно возвращалась по `id`.

In [ ]:
rows = connection.execute(
    """
    SELECT id, amount_cents, status
    FROM orders
    WHERE amount_cents >= ?
    ORDER BY amount_cents DESC, id ASC
    LIMIT ?
    """,
    (1000, 10),
).fetchall()
assert [dict(row) for row in rows] == [
    {"id": 2, "amount_cents": 2500, "status": "paid"},
    {"id": 3, "amount_cents": 1200, "status": "paid"},
]

> **Уточнено в Python 3.14.** Если SQL использует именованные placeholders вроде `:email`, `sqlite3` требует словарь параметров и поднимает `ProgrammingError` для последовательности. Для `?` передают последовательность. Это делает ошибочное смешение двух стилей явным; значения по-прежнему нельзя вставлять форматированием строк.

## `NULL` и трёхзначная логика

`NULL` означает отсутствие или неизвестность, а не пустую строку и не ноль. Сравнение `comment = NULL` даёт `UNKNOWN`, поэтому `WHERE` его не оставляет. Проверка записывается `IS NULL` или `IS NOT NULL`.

SQL-условия могут быть `TRUE`, `FALSE` и `UNKNOWN`. Например, `NOT (comment = 'x')` тоже не выберет строку с `NULL`: отрицание неизвестного остаётся неизвестным. Если пропуски надо считать отдельной категорией, это выражают явно.

In [ ]:
wrong = connection.execute(
    "SELECT id FROM orders WHERE comment = NULL"
).fetchall()
missing = connection.execute(
    "SELECT id FROM orders WHERE comment IS NULL ORDER BY id"
).fetchall()
assert wrong == []
assert [row["id"] for row in missing] == [1, 3]

## `JOIN`: связываем строки в запросе

`INNER JOIN` оставляет только пары, удовлетворяющие условию `ON`. `LEFT JOIN` сохраняет все строки слева; если пары справа нет, её столбцы становятся `NULL`.

Один клиент с двумя заказами даст две строки результата. Это не дубликат клиента в таблице, а две разные пары `клиент—заказ`. Если приложение хочет вложенный JSON, оно сгруппирует плоские строки после запроса или поручит это ORM.

In [ ]:
joined = connection.execute(
    """
    SELECT customers.email, orders.id AS order_id, orders.amount_cents
    FROM customers
    LEFT JOIN orders ON orders.customer_id = customers.id
    ORDER BY customers.email, orders.id
    """
).fetchall()
assert [(row["email"], row["order_id"]) for row in joined] == [
    ("anna@example.test", 1),
    ("anna@example.test", 2),
    ("boris@example.test", 3),
    ("vera@example.test", None),
]

Условие на правую таблицу после `WHERE` может незаметно убрать строки без пары. Запрос `LEFT JOIN orders ... WHERE orders.status = 'paid'` не вернёт Веру: для неё сравнение `NULL = 'paid'` неизвестно. Если нужно сохранить всех клиентов и присоединить только оплаченные заказы, условие переносится в `ON`: `LEFT JOIN orders ON ... AND orders.status = 'paid'`.

## Агрегаты, `GROUP BY` и `HAVING`

`COUNT`, `SUM`, `AVG`, `MIN` и `MAX` сворачивают набор строк. `GROUP BY customer_id` делает отдельную группу для каждого клиента. `WHERE` фильтрует строки **до** группировки, `HAVING` — готовые группы после вычисления агрегатов.

`COUNT(*)` считает строки результата, а `COUNT(column)` — только ненулевые значения столбца. После `LEFT JOIN` для клиента без заказов существует одна результирующая строка с `orders.id = NULL`: поэтому нужен `COUNT(orders.id)`, иначе получится один несуществующий заказ.

In [ ]:
stats = connection.execute(
    """
    SELECT
        customers.email,
        COUNT(orders.id) AS orders_count,
        COALESCE(SUM(CASE WHEN orders.status = 'paid' THEN orders.amount_cents ELSE 0 END), 0) AS paid_total
    FROM customers
    LEFT JOIN orders ON orders.customer_id = customers.id
    GROUP BY customers.id, customers.email
    HAVING COUNT(orders.id) >= ?
    ORDER BY paid_total DESC, customers.email
    """,
    (0,),
).fetchall()
assert [(row["email"], row["orders_count"], row["paid_total"]) for row in stats] == [
    ("anna@example.test", 2, 2500),
    ("boris@example.test", 1, 1200),
    ("vera@example.test", 0, 0),
]

## Транзакция: все изменения или ни одного

Перевод денег требует списать со счёта A и зачислить на B. Между этими действиями состояние временно неполно, поэтому они подтверждаются одним `COMMIT`. При любой ошибке `ROLLBACK` отменяет обе записи.

ACID кратко описывает свойства транзакций:

- atomicity — изменения подтверждаются вместе;
- consistency — ограничения остаются истинными после успешной операции;
- isolation — параллельные операции не используют недопустимое промежуточное состояние;
- durability — подтверждённые изменения сохраняются в рамках гарантий СУБД.

Конкретная изоляция и поведение блокировок зависят от СУБД и настроек.

In [ ]:
connection.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= 0))")
connection.executemany("INSERT INTO accounts (id, balance) VALUES (?, ?)", [(1, 1000), (2, 500)])
connection.commit()

In [ ]:
def transfer(connection: sqlite3.Connection, source_id: int, target_id: int, amount: int) -> None:
    if amount <= 0:
        raise ValueError("amount must be positive")
    with connection:
        source = connection.execute(
            "SELECT balance FROM accounts WHERE id = ?", (source_id,)
        ).fetchone()
        if source is None or source["balance"] < amount:
            raise ValueError("insufficient funds")
        updated = connection.execute(
            "UPDATE accounts SET balance = balance - ? WHERE id = ?",
            (amount, source_id),
        )
        target = connection.execute(
            "UPDATE accounts SET balance = balance + ? WHERE id = ?",
            (amount, target_id),
        )
        if updated.rowcount != 1 or target.rowcount != 1:
            raise ValueError("account not found")

transfer(connection, 1, 2, 300)
balances = connection.execute("SELECT balance FROM accounts ORDER BY id").fetchall()
assert [row["balance"] for row in balances] == [700, 800]

Контекстный менеджер соединения подтверждает транзакцию при нормальном выходе и откатывает при исключении, но **не закрывает соединение**. Это отличается от `with open(...)`. Соединение закрывают отдельным `connection.close()` или через `contextlib.closing`, когда закончен весь срок его жизни.

Транзакция должна соответствовать одной предметной операции. Коммит после каждой SQL-строки уничтожит атомарность; одна транзакция на всё время работы сервера будет держать ресурсы и блокировки слишком долго.

## Индекс: ускорение выбранного пути чтения

Без подходящего индекса база может просмотреть всю таблицу. Индекс хранит дополнительную упорядоченную структуру по выбранным столбцам и помогает быстро найти подходящие строки или уже нужный порядок. Индекс на внешнем ключе часто полезен для `JOIN` и каскадных проверок.

Цена индекса — место и дополнительная работа при `INSERT`, `UPDATE`, `DELETE`. Индекс создают под реальные запросы и проверяют план, а не добавляют на каждый столбец. В составном индексе важен порядок столбцов.

In [ ]:
connection.execute(
    "CREATE INDEX idx_orders_customer_status ON orders (customer_id, status)"
)
connection.commit()
plan = connection.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = ? AND status = ?",
    (1, "paid"),
).fetchall()
print([row["detail"] for row in plan])  # ищем использование idx_orders_customer_status

## Где уместна SQLite

SQLite хранит базу в одном файле и работает внутри процесса приложения. Она отлично подходит для локальных инструментов, прототипов, тестов, небольших сайтов и устройств. Ей не нужен отдельный сервер.

Но «один файл» не означает отсутствие конкурентности и транзакций. SQLite координирует доступ блокировками; профиль с множеством одновременных записей может потребовать серверную СУБД вроде PostgreSQL. Переезд не исправит плохую схему и запросы автоматически, зато изменит эксплуатацию, типы и детали SQL.

Модуль `sqlite3` синхронный: запрос внутри coroutine блокирует поток event loop. Соединение не следует бесконтрольно делить между потоками и задачами; веб-приложение задаёт понятный срок жизни соединения или session на запрос и использует интеграцию фреймворка для блокирующего I/O.

## SQLAlchemy и SQLModel: слой над теми же понятиями

ORM связывает таблицу с классом, строку — с объектом, столбец — с атрибутом. Engine знает, как подключаться к базе и управлять пулом соединений. Session ограничивает разговор с базой и транзакцию, отслеживает объекты. `select()` строит запрос.

SQLModel объединяет модели Pydantic и SQLAlchemy, что удобно рядом с FastAPI. Но ORM не отменяет внешние ключи, транзакции, планы запросов и SQL: она генерирует SQL и может сгенерировать неудачный запрос, если модель или способ загрузки выбраны плохо.

In [ ]:
from sqlmodel import Field, Session, SQLModel, create_engine, select

class CustomerModel(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    email: str = Field(index=True, unique=True)
    name: str

engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)

with Session(engine) as session:
    session.add(CustomerModel(email="anna@example.test", name="Анна"))
    session.commit()

with Session(engine) as session:
    statement = select(CustomerModel).where(CustomerModel.email == "anna@example.test")
    customer = session.exec(statement).one()
    assert customer.name == "Анна"

Session не является глобальным кэшем приложения и не должна разделяться произвольно между запросами. Типичный веб-запрос получает собственную session, выполняет одну предметную операцию, подтверждает или откатывает транзакцию и закрывает session. Engine, наоборот, обычно создаётся один раз на приложение.

`session.add()` ещё не означает сохранение в базе. `flush` отправляет изменения в текущую транзакцию, `commit` подтверждает её, `refresh` перечитывает значения вроде сгенерированного id. Видимый Python-объект не отменяет границы транзакции.

## `create_all` — не система миграций

`SQLModel.metadata.create_all(engine)` удобно создаёт отсутствующие таблицы для учебного примера. Оно не превращает существующую production-схему в новую версию модели: не планирует переименование столбца, перенос данных и безопасный откат.

Изменения рабочей схемы оформляют версионированными миграциями, обычно через Alembic в экосистеме SQLAlchemy. Миграция содержит порядок изменения структуры и, при необходимости, преобразование существующих данных. Файл базы не коммитят как замену миграциям.

## Неожиданно, но по правилам

### 1. `NULL = NULL` не истинно

Оба значения неизвестны и не обязаны быть одинаковыми. Для проверки отсутствия нужен `IS NULL`.

### 2. `COUNT(*)` после `LEFT JOIN` может дать один заказ вместо нуля

Он считает сохранённую строку клиента с пустой правой частью. `COUNT(orders.id)` игнорирует её `NULL`.

### 3. `LEFT JOIN` легко превратить в `INNER JOIN` условием `WHERE`

Фильтр `WHERE orders.status = 'paid'` удаляет клиентов без заказа. Чтобы сохранить их, условие присоединяемой таблицы часто помещают в `ON`.

### 4. Объявленный внешний ключ SQLite может ничего не проверять

`PRAGMA foreign_keys = ON` включается отдельно для каждого соединения и не меняется посреди транзакции. Поэтому приложение включает и проверяет его сразу после подключения.

### 5. `with connection:` не закрывает соединение

Он управляет подтверждением или откатом транзакции. После блока соединение остаётся доступным.

### 6. Результат без `ORDER BY` не имеет обещанного порядка

Совпадение с порядком вставки на маленьких данных — наблюдение, не контракт. Индекс, план или версия могут изменить выдачу.

### 7. Placeholder не заменяет имя столбца

`ORDER BY ?` сортирует по переданному значению-константе, а не выбирает столбец. Допустимое имя выбирают в Python из белого списка и добавляют как проверенную часть SQL.

### 8. Индекс способен замедлить приложение

Каждая запись обновляет индекс, а неподходящий индекс занимает место и может не использоваться планировщиком.

### 9. ORM-объект не означает, что изменение сохранено

До `commit` транзакция может быть откатана. После закрытия session лениво загружаемая связь может быть недоступна.

### 10. Уникальность и `NULL` сочетаются не всегда интуитивно

В SQLite `UNIQUE` допускает несколько строк с `NULL`, потому что неизвестные значения не считаются равными. Если поле обязательно уникально и обязательно заполнено, нужны одновременно `UNIQUE` и `NOT NULL`.

## Самопроверка

1. Когда таблица лучше JSON-файла?
2. Что представляет одна строка каждой нашей таблицы?
3. Зачем заказу внешний ключ?
4. Чем `PRIMARY KEY` отличается от `UNIQUE` email?
5. Почему ограничения дублируют часть проверок приложения?
6. Зачем значения передавать placeholders?
7. Почему placeholder не подходит для имени таблицы?
8. Как проверить `NULL`?
9. Чем `INNER JOIN` отличается от `LEFT JOIN`?
10. Почему после связи «один ко многим» клиент повторяется в результате?
11. Чем `WHERE` отличается от `HAVING`?
12. Почему для клиента без заказов нужен `COUNT(orders.id)`?
13. Что гарантирует транзакция перевода?
14. Что появилось в `sqlite3` Python 3.12?
15. Зачем включать `PRAGMA foreign_keys`?
16. Какова цена индекса?
17. Чем engine отличается от session?
18. Почему `create_all` не заменяет миграции?

## Источники

- [Документация `sqlite3` Python 3.14](https://docs.python.org/3.14/library/sqlite3.html) — соединения, placeholders, транзакции и `row_factory`.
- [Язык SQLite](https://www.sqlite.org/lang.html) — синтаксис SQL.
- [Внешние ключи SQLite](https://www.sqlite.org/foreignkeys.html) — включение и проверка связей.
- [Планировщик запросов SQLite](https://www.sqlite.org/queryplanner.html) — назначение индексов.
- [SQLAlchemy Unified Tutorial](https://docs.sqlalchemy.org/en/20/tutorial/) — современный стиль SQLAlchemy 2.x.
- [SQLModel: создание таблиц](https://sqlmodel.tiangolo.com/tutorial/create-db-and-table/) и [чтение данных](https://sqlmodel.tiangolo.com/tutorial/select/) — engine, session и `select()`.

Конкретные ограничения и каскады в проекте выбираются по предметной области, а не копируются из демонстрационной схемы.

## Итоги

- Реляционная схема хранит отдельные факты в таблицах и связывает их ключами.
- Ограничения защищают базовые инварианты независимо от пути записи.
- SQL декларативно задаёт нужный результат; значения передаются параметрами.
- `NULL` участвует в трёхзначной логике и проверяется через `IS NULL`.
- `JOIN` строит пары строк, агрегаты сворачивают группы.
- Транзакция подтверждает связанные изменения вместе или откатывает их.
- Индекс ускоряет выбранные запросы ценой места и записи.
- В Python 3.14 режим `sqlite3.autocommit` лучше задавать явно; внешние ключи SQLite включаются на соединении.
- ORM отображает те же таблицы и транзакции на объекты, но не заменяет знание SQL.
- `create_all` подходит для старта учебной базы, миграции — для изменения существующей.

На семинаре построим схему в памяти, выполним запросы, проверим rollback и посмотрим план индекса.